# WSJ 2027 - Gruppindelning Direktresa

Assign confirmed direktresa deltagare into groups of exactly 36.

Direktresa participants travel independently to/from WSJ in Poland.
Same grouping algorithm as rundresa but applied to the direktresa subset.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort, friend-cluster-aware
3. **Friend wish** — at least ONE friend in same group (soft goal)
4. **Max 6 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex balance


In [1]:
TRAVEL = 'direktresa'        # only line that differs between rundresa.ipynb and direktresa.ipynb
QUALITY = 'slow'           # 'medium' (~1-3 min) or 'slow' (~10-30 min, +5-10% friends)
GROUP_SIZE = 36
MAX_KAR = 6                # max participants from same kår per group
OUTPUT_DIR = '/config/notebooks/wsj27/output'


In [ ]:
import sys; sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

raw = u.fetch_participants()
df_all, _ = u.build_participant_dataframe(raw)
df = df_all[(df_all['travel'] == TRAVEL) & (df_all['category'] == 'deltagare')].copy().reset_index(drop=True)
u.assign_coordinates(df)
df = u.add_hilbert_index(df)
u.resolve_friend_wishes(df, df_all)
u.apply_manual_overrides(df, df_all)
friend_wishes = u.build_friend_graph(df)
u.print_intake_summary(df, GROUP_SIZE)


In [ ]:
df = u.assign_groups(df, GROUP_SIZE, friend_wishes, max_kar=MAX_KAR, quality=QUALITY)
u.apply_manual_placement(df)
group_of = df['group'].values
total_groups = df['group'].nunique()



############################################################
# Slow tier: 8 restarts
############################################################

----- Restart 1/8 (seed=42) -----
Participants: 568
Groups: 15 x 36 + 1 x 28 = 16 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 337/337
  Kar violations: 32
  Avg geo spread: 3.8960

=== Phase 2: Fix friend wishes (criticality-ordered, iterate) ===
  No unsatisfied wishes; converged after 0 pass(es)
  Total swaps: 0
  Friend satisfaction: 337/337
  Kar violations: 32
  Avg geo spread: 3.8960

=== Phase 2.5: 3-way rotations ===
  Rotations: 0
  Friend satisfaction: 337/337
  Kar violations: 32

=== Phase 3: Fix kar violations (friend-aware) ===
  Swaps: 32
  Kar violations: 0
  Friend satisfaction: 324/337
  Avg geo spread: 3.9174

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 10
  Friend satisfaction: 334/337
  Kar violations: 0
  Avg geo spread: 3.9027

=== Phase 4: Weighted SA (friend-positive) ===
  Swaps

In [ ]:
u.print_group_metrics(df, group_of, total_groups)
csv_path, json_path = u.export_results(df, group_of, total_groups, OUTPUT_DIR, prefix=f'wsj27_{TRAVEL}')
map_path = f'{OUTPUT_DIR}/wsj_{TRAVEL}_karta.html'
u.generate_group_map_html(df, total_groups, map_path,
                          title=f'WSJ 2027 {TRAVEL.title()}',
                          friend_wishes=friend_wishes, show_group_arcs=True)
report_path = f'{OUTPUT_DIR}/wsj_{TRAVEL}_grupper.html'
u.generate_groups_report_html(df, total_groups, report_path,
                              title=f'WSJ 2027 {TRAVEL.title()} - Grupper',
                              friend_wishes=friend_wishes,
                              group_size=GROUP_SIZE,
                              max_kar=MAX_KAR,
                              quality=QUALITY)
print(f'CSV:    {csv_path}\nJSON:   {json_path}\nMap:    {map_path}\nReport: {report_path}')
